# Analyzing a CBED Pattern — Silicon [001]

**CITEAM 2027 training module (draft) — Part D: Analyze**

The morning notebook built a diffraction pattern from first principles with *parallel*
illumination: every reflection was a **point**. This notebook converges the beam into a
cone. Each reflection opens up into a **disk**, and each point inside that disk is a
different incident direction — so a CBED disk is a two-dimensional *map of the excitation
error* $s_g$ we derived this morning.

That buys two things the morning pattern could not give:

* **Geometry, more precisely.** Disk positions still obey the kinematic rules from Part B,
  but the disk edges are sharp features that calibrate better than diffuse spots.
* **Dynamical information.** The fringes *inside* each disk are a rocking curve. They
  encode specimen thickness and the extinction distance — quantities that are simply
  invisible in a kinematic spot pattern.

We work on a real CBED pattern of silicon recorded on a Zeiss Libra 200 MC, and we end
where Part B ended: by writing out a dataset that describes itself.

The pedagogical structure follows, with permission and gratitude, the *Analyzing CBED
Pattern* notebook by **Gerd Duscher and Sergei Kalinin** (MLSTEM 2026 workshop, University
of Tennessee, Knoxville — [github.com/gduscher/MLSTEM2026](https://github.com/gduscher/MLSTEM2026)),
reimplemented here on the **HyperSpy ecosystem** (`hyperspy`, `diffsims`, `ase`) used
throughout CITEAM. The data file is the one used in that workshop.

**Content**
1. The file, and what it claims
2. Electron wavelength and the camera constant
3. Why a reflection becomes a disk
4. Finding the centre and the disk lattice
5. Calibration: nominal camera length vs. the specimen
6. Kinematic simulation with `diffsims`
7. Overlay: indexing the experimental pattern
8. Inside the disk: where kinematic theory ends
9. Data-management epilogue: a dataset that corrects its own file


In [ ]:
import sys, importlib.metadata

def test_package(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return '-1'

for pkg in ['hyperspy', 'diffsims', 'ase', 'orix']:
    if test_package(pkg) == '-1':
        print(f'installing {pkg}')
        !{sys.executable} -m pip install {pkg} -q
print('done')

In [ ]:
%matplotlib inline
# On Google Colab / JupyterLab with ipympl installed, '%matplotlib widget'
# gives interactive (zoomable) figures. 'inline' guarantees static figures
# everywhere and keeps the executed notebook archivable.
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import scipy.constants as const
from scipy import ndimage

import ase, ase.build
import hyperspy.api as hs
import diffsims

# Show version numbers for provenance / archiving.
__notebook_version__ = '2026.10.08-CITEAM-draft'
print('notebook version:', __notebook_version__)
for m in (np, matplotlib, ase, hs, diffsims):
    print(f'{m.__name__:12s}', m.__version__)

## 1. The file, and what it claims

The pattern is `CBED_375mm_alpha-3.3_beta9.6_Condensor10um_Filter.dm3` from the MLSTEM
example data. It is a Digital Micrograph file, so — unlike the orphan TIF in Part B — it
arrives *full* of metadata.

That turns out to be the harder problem. An empty file announces its emptiness. A file
full of confident, wrong metadata does not.

In [ ]:
import os, urllib.request

data_file = 'data/CBED_375mm_alpha-3.3_beta9.6_Condensor10um_Filter.dm3'
url = ('https://github.com/gduscher/MLSTEM2026/raw/main/example_data/'
       'CBED_375mm_alpha-3.3_beta9.6_Condensor10um_Filter.dm3')

if not os.path.exists(data_file):
    os.makedirs('data', exist_ok=True)
    print('downloading ~17 MB from the MLSTEM2026 repository ...')
    urllib.request.urlretrieve(url, data_file)

s = hs.load(data_file)
print(s)
print(s.axes_manager)

Read the axes again.

This is a **diffraction pattern**. Its axes should be reciprocal — $\mathrm{nm}^{-1}$ or
mrad. Digital Micrograph has stamped it with a **real-space** scale, $0.403\ \mathrm{nm}$
per pixel, because the microscope was left flagged in imaging mode when the camera was
read out. Anyone who trusts `axes_manager` here measures the lattice of a crystal in
nanometres and gets a number that is wrong by orders of magnitude, with no warning at all.

Now look at what else the file asserts.

In [ ]:
om = s.original_metadata.as_dictionary()
mi  = om['ImageList']['TagGroup0']['ImageTags']['Microscope Info']
acq = om['ImageList']['TagGroup0']['ImageTags']['Acquisition']

print('--- what the file gets RIGHT (real, recorded instrument physics) ---')
print(f"  high tension          : {mi['Voltage']:.0f} V")
print(f"  camera length (nominal): {mi['STEM Camera Length']:.1f} mm")
print(f"  Cs                    : {mi['Cs(mm)']} mm")
print(f"  microscope            : {mi['Microscope']}")
print(f"  camera                : {acq['Device']['Name']}")
print(f"  CCD pixel size        : {acq['Device']['CCD']['Pixel Size (um)']} um")
print(f"  binning               : {acq['Frame']['Area']['Transform']['Transform List']['TagGroup0']['Binning']}")
print(f"  exposure              : {acq['Parameters']['Detector']['exposure (s)']} s")
print(f"  processing            : {acq['Parameters']['High Level']['Processing']}")

print('\n--- what the file gets WRONG ---')
print(f"  operation mode        : {mi['Operation Mode']}      <- it is a DIFFRACTION pattern")
print(f"  imaging mode          : {mi['Imaging Mode']}")
print(f"  indicated magnification: {mi['Indicated Magnification']}   <- meaningless in diffraction")
print(f"  axis calibration      : {s.axes_manager[0].scale:.4f} {s.axes_manager[0].units} per pixel"
      f"   <- real space, on a diffraction pattern")
print(f"  specimen field        : {mi['Specimen']!r}")
print('                          ^ a block copolymer. This pattern is a crystal.')
print(f"  operator              : {mi['Operator']!r}   (stale from an earlier session)")

Three different kinds of failure, in one file:

| what | status | consequence |
|---|---|---|
| high tension, camera length, CCD pixel size | **correct, recorded** | we can calibrate from physics |
| mode, magnification, axis scale | **wrong** — imaging values on a diffraction pattern | silently wrong lengths |
| specimen, operator | **stale** — left over from a previous session | wrong sample attribution |

And the only place the acquisition conditions were actually recorded is the **file name**:
`375mm`, `alpha-3.3`, `beta9.6`, `Condensor10um`, `Filter`. A file name is not metadata. It
cannot be queried, it is not validated, it does not survive a rename — and in section 5 we
will find that even its camera length disagrees with the specimen.

The good news: the three numbers we need to calibrate a diffraction pattern from first
principles — **high tension, camera length, physical pixel size** — are all genuinely in
the file. We will use them, then let the crystal correct them.

In [ ]:
img = np.asarray(s.data, float)
img -= img.min()          # gain-normalised DM data carries a dark-level offset

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('linear intensity')
axes[1].imshow(np.log1p(img), cmap='gray')
axes[1].set_title('log intensity — the disks appear')
for ax in axes:
    ax.set_xlabel('pixel'); ax.set_ylabel('pixel')
plt.tight_layout()

print(f'dynamic range: {img.min():.0f} to {img.max():.0f} counts')

The direct beam saturates everything on a linear scale. Diffraction data spans a far wider
dynamic range than an image, so we display it logarithmically throughout — `log1p` is
$\log(1+I)$, which is well behaved at zero and needs no fudge constant.

## 2. Electron wavelength and the camera constant

Same relativistic wavelength as Part B — the physics has not changed, only the
illumination:

$$\lambda = \frac{h}{\sqrt{2 m_e eV\left(1 + \frac{eV}{2 m_e c^2}\right)}}$$

What is new is how we get from a *pixel* to a *scattering vector*. On a detector a distance
$L$ (the camera length) behind the specimen, a beam diffracted through $2\theta$ lands at
radius

$$r = L \cdot 2\theta , \qquad 2\theta = \lambda g \ \ \text{(Bragg, small angle)}$$

so $r = L\lambda g$. The product $L\lambda$ is the **camera constant**, and a detector
pixel of physical size $p$ subtends

$$\boxed{\ \Delta g = \frac{p}{L\lambda}\ }$$

This is the honest replacement for a hand-tuned `pixel_size`. Every quantity on the right
is recorded in the file.

In [ ]:
def get_wavelength(acceleration_voltage_V):
    '''Relativistic electron wavelength in metres.'''
    V = acceleration_voltage_V
    momentum = np.sqrt(2 * const.m_e * const.e * V *
                       (1 + const.e * V / (2 * const.m_e * const.c**2)))
    return const.h / momentum

acceleration_voltage_V = mi['Voltage']                                   # 200 kV, from the file
camera_length_nominal_m = mi['STEM Camera Length'] * 1e-3                # mm -> m, from the file
pixel_size_m = acq['Device']['CCD']['Pixel Size (um)'][0] * 1e-6         # um -> m, from the file

wave_length_m = get_wavelength(acceleration_voltage_V)
print(f'wavelength            = {wave_length_m*1e12:.5f} pm')
print(f'camera constant Lλ    = {camera_length_nominal_m*wave_length_m*1e9*1e3:.4f} mm·nm')

# cross-check against the diffsims implementation (which takes kV and returns Angstrom)
from diffsims.utils.sim_utils import get_electron_wavelength
assert np.isclose(wave_length_m*1e10,
                  get_electron_wavelength(acceleration_voltage_V/1000), rtol=1e-6)
print('matches diffsims.utils.sim_utils.get_electron_wavelength')

dg_nominal = pixel_size_m / (camera_length_nominal_m * wave_length_m) / 1e9   # 1/m -> 1/nm
print(f'\nnominal scale         = {dg_nominal:.6f} 1/nm per pixel')
print(f'  from L = {camera_length_nominal_m*1e3:.0f} mm, p = {pixel_size_m*1e6:.0f} um, '
      f'λ = {wave_length_m*1e12:.3f} pm')

Hold on to that number. It comes from instrument physics alone — no crystal, no fitting,
no scale bar. In section 5 we will ask the silicon whether it is true.

*(For comparison: the MLSTEM notebook this is based on sets `pixel_size = 20e-4` by hand
and tells the student to adjust it until the overlay looks right. That works, but it hides
the fact that the file already knows almost the whole answer.)*

## 3. Why a reflection becomes a disk

In Part B the illumination was a plane wave: one incident wavevector $\vec{K}_0$, so for
each reciprocal-lattice point $\vec g$ there was exactly one excitation error $s_g$, and
the reflection was a **point**.

Converging the beam replaces that single $\vec{K}_0$ with a *cone* of incident directions
of semi-angle $\alpha$. Every direction in the cone diffracts off the same $\vec g$, but
each arrives at the detector slightly displaced — so the reflection spreads into a **disk**
of angular radius $\alpha$, i.e. a radius in reciprocal space of

$$k_\alpha = \frac{\alpha}{\lambda}$$

The important consequence is not that the spot got fatter. It is that **position inside the
disk is meaningful**: each point corresponds to one incident direction, hence to one value
of the excitation error. Scanning across a disk is physically a *rocking curve* — the same
curve you would measure by tilting the crystal, acquired in parallel in a single exposure.

For a tilt $\Delta\theta$ away from the exact Bragg condition, measured along $\vec g$,

$$s_g \simeq -g\,\Delta\theta = -\lambda\, g\, \Delta k$$

where $\Delta k$ is the displacement from the Bragg position *within* the disk, in
$\mathrm{nm}^{-1}$. We use this in section 8.

Two regimes matter:

* **Kossel–Möllenstedt**, $k_\alpha < g/2$: disks stay separate. Each disk is an
  independent rocking curve. This is the useful regime for thickness measurement.
* **Kossel**, $k_\alpha > g/2$: disks overlap and interfere.

Our pattern sits essentially *on* that boundary, as we are about to measure.

## 4. Finding the centre and the disk lattice

The MLSTEM notebook does this with a mouse: an `EllipseSelector` widget that the student
drags over the central disk. That is excellent in a live classroom and impossible
everywhere else — it cannot run headless, it cannot be re-executed, and the number it
produces is never written down.

We do it numerically instead, so the result is reproducible and archivable.

### Finding the centre without touching a disk

The obvious recipe — "the direct beam is the brightest disk" — **fails on this pattern**. A
strongly excited $\{220\}$ disk outshines the transmitted one here, which is perfectly
normal in a zone-axis CBED pattern of a thick crystal: intensity has flowed out of the
direct beam into the Bragg reflections. Picking the brightest disk gets you a first-order
disk, a centre that is wrong by one lattice vector, and a scale that is wrong by $\sqrt2$.

Use symmetry instead. Friedel's law makes a diffraction pattern centrosymmetric about the
direct beam: $I(\vec c + \vec u) = I(\vec c - \vec u)$. So the centre is the point about
which the pattern maps onto its own 180° rotation — and the correlation of the image with
its own inversion peaks at exactly $2\vec c$. That uses every pixel in the pattern, not one
disk, and it does not care which disk is brightest.

In [ ]:
window = np.hanning(img.shape[0])[:, None] * np.hanning(img.shape[1])[None, :]
F = np.fft.fft2(img * window)
autoconvolution = ndimage.gaussian_filter(np.real(np.fft.ifft2(F * F)), 3)
iy, ix = np.unravel_index(np.argmax(autoconvolution), autoconvolution.shape)

# the peak sits at 2*centre, modulo the array size -> four candidates; take the
# one with the most intensity around it
candidates = [((ix/2) % img.shape[1] + dx, (iy/2) % img.shape[0] + dy)
              for dx in (0, img.shape[1]/2) for dy in (0, img.shape[0]/2)]
def enclosed_intensity(c, r=200):
    yy, xx = np.ogrid[:img.shape[0], :img.shape[1]]
    return img[(yy - c[1])**2 + (xx - c[0])**2 < r**2].sum()
centre_x, centre_y = max(candidates, key=enclosed_intensity)
print(f'direct beam by Friedel symmetry: (x={centre_x:.1f}, y={centre_y:.1f}) px')

brightest_disk = np.unravel_index(np.argmax(ndimage.gaussian_filter(img, 25)), img.shape)
print(f'brightest disk is at             (x={brightest_disk[1]}, y={brightest_disk[0]}) px'
      '   <- NOT the direct beam')

Now the disks themselves:

1. divide out the broad radial fall-off, so faint outer disks weigh as much as bright inner ones;
2. smooth away the fringes *inside* the disks, which are not what we are locating;
3. take local maxima — these are the disk centres;
4. index them against a basis and refine centre and lattice vectors by least squares.

For step 4 the basis must be the two shortest non-collinear vectors — a *reduced* basis. Take
any two first-shell disks that happen to be detected and you may pick a pair spanning the
$\{400\}$ diagonals instead, which indexes just as consistently and is wrong by $\sqrt2$.

In [ ]:
# ---------------- INPUT ----------------
smooth_background = 220   # px, scale of the radial fall-off to divide out
smooth_disk       = 18    # px, smooths the fringes inside a disk
min_separation    = 150   # px, no two disk centres closer than this
# ---------------------------------------

background = ndimage.gaussian_filter(img, smooth_background) + 1.0
normalised = ndimage.gaussian_filter(img / background, smooth_disk)
local_max = ndimage.maximum_filter(normalised, min_separation)
peaks = (normalised == local_max) & (normalised > np.percentile(normalised, 98.5))
peak_y, peak_x = np.nonzero(peaks)
print(f'{len(peak_y)} disk centres found')

relative = np.stack([peak_x - centre_x, peak_y - centre_y], 1).astype(float)
distance = np.hypot(*relative.T)

# reduced basis: shortest vector, then the shortest one not parallel to it
order = np.argsort(distance)
order = order[distance[order] > 20]
a1_trial = relative[order[0]]
a2_trial = None
for i in order[1:]:
    cos = abs(relative[i] @ a1_trial /
              np.linalg.norm(relative[i]) / np.linalg.norm(a1_trial))
    if cos < 0.9:
        a2_trial = relative[i]
        break
print(f'reduced basis: a1 = {a1_trial.round(1)} (|a1| = {np.linalg.norm(a1_trial):.1f} px),'
      f'  a2 = {a2_trial.round(1)} (|a2| = {np.linalg.norm(a2_trial):.1f} px)')

basis = np.stack([a1_trial, a2_trial], 1)
mn = np.linalg.solve(basis, relative.T).T
on_lattice = np.all(np.abs(mn - np.round(mn)) < 0.20, 1)
mn_int, observed = np.round(mn[on_lattice]), relative[on_lattice]
print(f'{on_lattice.sum()} disks indexed:',
      [tuple(int(v) for v in r) for r in mn_int])

# least squares for (a1x,a2x,a1y,a2y,dx,dy)
A = np.zeros((2*len(mn_int), 6)); b = observed.reshape(-1)
A[0::2, 0], A[0::2, 1], A[0::2, 4] = mn_int[:, 0], mn_int[:, 1], 1
A[1::2, 2], A[1::2, 3], A[1::2, 5] = mn_int[:, 0], mn_int[:, 1], 1
sol, *_ = np.linalg.lstsq(A, b, rcond=None)

a1 = np.array([sol[0], sol[2]]); a2 = np.array([sol[1], sol[3]])
centre = np.array([centre_x + sol[4], centre_y + sol[5]])
residual = np.hypot(*(A @ sol - b).reshape(-1, 2).T)

print(f'\nrefined centre  = ({centre[0]:.1f}, {centre[1]:.1f}) px')
print(f'refined a1      = {a1.round(1)}  |a1| = {np.linalg.norm(a1):.2f} px'
      f'  at {np.degrees(np.arctan2(-a1[1], a1[0])):.2f} deg')
print(f'refined a2      = {a2.round(1)}  |a2| = {np.linalg.norm(a2):.2f} px'
      f'  at {np.degrees(np.arctan2(-a2[1], a2[0])):.2f} deg')
included = np.degrees(np.arccos(a1 @ a2 / np.linalg.norm(a1) / np.linalg.norm(a2)))
print(f'included angle  = {included:.2f} deg       (90 deg = undistorted square net)')
print(f'rms residual    = {residual.mean():.2f} px')

g_first_px = 0.5*(np.linalg.norm(a1) + np.linalg.norm(a2))
print(f'\nfirst-order spacing = {g_first_px:.2f} px')

fig, ax = plt.subplots(figsize=(7.5, 7.5))
ax.imshow(np.log1p(img), cmap='gray')
ax.plot(peak_x, peak_y, 'c+', ms=14, mew=1.6, label='detected disks')
ax.plot(*centre, 'r+', ms=22, mew=2.5, label='direct beam (Friedel)')
ax.plot(*brightest_disk[::-1], 'yx', ms=16, mew=2.5, label='brightest disk')
for v, c in ((a1, 'lime'), (a2, 'magenta')):
    ax.annotate('', xy=centre+v, xytext=centre,
                arrowprops=dict(color=c, width=1.8, headwidth=9))
ax.legend(loc='upper right', fontsize=9)
ax.set_title('disk detection, refined centre and reduced basis')
plt.tight_layout()

Two things to read off that output.

**The net is square** — the included angle is within a degree or two of 90°, and the two
lattice vectors have nearly equal length. A square zero-order Laue zone is the signature of
a $\langle 001\rangle$ zone axis in a cubic crystal, which is what the MLSTEM notebook
assumes and what we will simulate.

**It is not quite square.** The small residual difference between $|a_1|$ and $|a_2|$, and
the departure of the included angle from 90°, are real: the projector lenses of a TEM image
reciprocal space with a slight elliptical distortion and a rotation. We keep the mean of the
two lengths as our scale and note the anisotropy rather than pretending it is zero.

Before using this lattice, we check that it is the *first* shell and not a coarser
sub-lattice of a finer one. If there were disks halfway between ours, our scale would be
wrong by a factor of two — and the overlay in section 7 would still look convincing.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7.5))
for ax, (title, step) in zip(axes, [('lattice as refined', 1.0),
                                    ('half-spacing test', 0.5)]):
    ax.imshow(np.log1p(img), cmap='gray')
    for m in range(-4, 5):
        for n in range(-4, 5):
            p = centre + (m*a1 + n*a2) * step
            if not (0 < p[0] < img.shape[1] and 0 < p[1] < img.shape[0]):
                continue
            integer = (abs(m*step % 1) < 1e-6) and (abs(n*step % 1) < 1e-6)
            ax.plot(*p, 'o' if integer else 'x',
                    mfc='none', mec='cyan' if integer else 'orange',
                    ms=15 if integer else 9, mew=1.6)
    ax.set_title(title)
axes[1].set_title('half-spacing test: orange = half-integer positions')
plt.tight_layout()
print('Cyan lands on disks. If orange also landed on disks, our lattice would be')
print('the SECOND shell and every length below would be wrong by a factor of 2.')

The half-integer positions fall in the gaps. Our lattice is the first shell of allowed
reflections.

Finally the disk radius, which gives the convergence angle. We take it from the steepest
drop in the radial profile of the direct disk — the disk edge is the sharpest feature in a
CBED pattern, which is exactly why CBED calibrates well.

In [ ]:
yy, xx = np.indices(img.shape)
radius_px = np.hypot(yy - centre[1], xx - centre[0]).astype(int)
profile = ndimage.mean(img, radius_px, index=np.arange(int(g_first_px)))
smoothed = ndimage.gaussian_filter1d(profile, 4)
# search between 0.15 and 0.80 of the lattice spacing - tied to the measured
# geometry rather than to magic pixel numbers
lo, hi = int(0.15*g_first_px), int(0.80*g_first_px)
disk_radius_px = int(np.argmin(np.gradient(smoothed)[lo:hi])) + lo

print(f'disk radius = {disk_radius_px} px   (searched {lo}-{hi} px)')
print(f'disk radius / first-order spacing = {disk_radius_px/g_first_px:.3f}')
print('  (0.5 is the Kossel-Moellenstedt / Kossel boundary — disks just touching)')

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(profile, label='radial mean of the direct disk')
ax.axvline(disk_radius_px, color='r', ls='--', label=f'edge, {disk_radius_px} px')
ax.set_xlabel('radius (px)'); ax.set_ylabel('counts'); ax.legend()
ax.set_title('the disk edge is the sharpest feature in the pattern')
plt.tight_layout()

## 5. Calibration: nominal camera length vs. the specimen

Now the two independent routes to the scale meet.

**Route 1 — instrument physics (section 2).** The file says $L = 378\ \mathrm{mm}$,
$p = 14\ \mathrm{\mu m}$, $200\ \mathrm{kV}$, giving $\Delta g = p/(L\lambda)$.

**Route 2 — the crystal.** Silicon is diamond cubic, $a = 5.4309\ \mathrm{Å}$. In the
$[001]$ zone the structure factor kills $\{200\}$ (we re-derive this in section 6), so the
**first allowed reflection of the square net is $\{220\}$**, at
$g_{220} = \sqrt{8}/a$. We measured that spacing in pixels above.

A crystal is a better ruler than a lens current. Route 2 is the calibration we keep; the
discrepancy with route 1 is the lesson.

In [ ]:
# ---------------- INPUT ----------------
lattice_parameter_nm = 0.54309      # silicon, ICSD
first_allowed_hkl    = (2, 2, 0)    # first allowed reflection of the [001] ZOLZ
# ---------------------------------------

g_first_invnm = np.sqrt(np.sum(np.array(first_allowed_hkl)**2)) / lattice_parameter_nm
scale_refined = g_first_invnm / g_first_px

print(f'g_{"".join(map(str, first_allowed_hkl))} = {g_first_invnm:.4f} 1/nm'
      f'   measured at {g_first_px:.2f} px')
print(f'\n  nominal scale (camera length) = {dg_nominal:.6f} 1/nm per px')
print(f'  refined scale (silicon)       = {scale_refined:.6f} 1/nm per px')
print(f'  disagreement                  = {100*(dg_nominal/scale_refined - 1):+.1f} %')

camera_length_true_m = pixel_size_m / (scale_refined*1e9 * wave_length_m)
print(f'\n  camera length, nominal (file + file name) = '
      f'{camera_length_nominal_m*1e3:.0f} mm  /  375 mm')
print(f'  camera length, implied by the silicon     = {camera_length_true_m*1e3:.0f} mm')

convergence_invnm = disk_radius_px * scale_refined
convergence_mrad  = convergence_invnm * wave_length_m*1e9 * 1e3
print(f'\n  convergence semi-angle, measured = {convergence_mrad:.2f} mrad'
      f'   (disk radius {convergence_invnm:.3f} 1/nm)')
print(f'  convergence semi-angle, file name = 3.3 mrad')

This is the heart of the notebook, so it is worth being explicit about what just happened.

**The nominal camera length is wrong by 17%.** 378 mm from the metadata (375 mm in the
file name) against about 324 mm implied by the silicon. This is not a defect in
this particular microscope — nominal camera lengths are lens-current set points, and the
*actual* projector magnification depends on the lens history, the eucentric height and the
excitation of everything upstream. Nominal camera lengths are routinely off by 10–20%. They
are a starting value, never a calibration.

**The convergence angle in the file name is wrong too**, by roughly a factor of two against
the measured disk radius. `alpha-3.3` and `beta9.6` are the illumination and collection
semi-angles of the energy-filter setup on this instrument; they were carried into the file
name by a naming habit and do not describe the convergence angle that actually formed these
disks. The disks themselves do.

**So: every acquisition parameter recorded only as text was wrong, and every parameter
recorded as data was right.** The camera length and pixel size got us within 15% without
touching the specimen — which is close enough to index the pattern unambiguously, and that
is exactly what a nominal value is for. The crystal then supplied the rest.

That 15% matters. A lattice parameter measured on the nominal scale would be wrong in the
first significant figure, which is worse than useless — it is publishable-looking and
false. We carry the refined scale from here on.

In [ ]:
scale = scale_refined          # 1/nm per pixel, used from here on

# in-plane rotation between the simulation frame and the detector
rotation_measured_deg = np.degrees(np.arctan2(-a1[1], a1[0])) - 45.0
print(f'measured orientation of a1 = {np.degrees(np.arctan2(-a1[1], a1[0])):.2f} deg')
print(f'simulated {{220}} will sit at 45 deg, so rotation = '
      f'{rotation_measured_deg:.2f} deg')

print(f'\ncalibration in force:')
print(f'  scale            = {scale:.6f} 1/nm per pixel')
print(f'  centre           = ({centre[0]:.1f}, {centre[1]:.1f}) px')
print(f'  rotation         = {rotation_measured_deg:.2f} deg')
print(f'  convergence      = {convergence_invnm:.3f} 1/nm ({convergence_mrad:.2f} mrad)')
print(f'  field of view    = {img.shape[1]*scale:.2f} x {img.shape[0]*scale:.2f} 1/nm')

## 6. Kinematic simulation with `diffsims`

The disk *positions* are pure kinematics — exactly the Part B machinery, so we reuse it
rather than re-deriving it. Silicon is diamond cubic: an fcc lattice with a two-atom basis
at $(0,0,0)$ and $(\frac14,\frac14,\frac14)$, giving

$$F_{hkl} = f_{\mathrm{Si}} \sum_{i \in \mathrm{u.c.}} e^{-2\pi i \vec g \cdot \vec r_i}
 = 4 f_{\mathrm{Si}} \left[1 + e^{-\frac{\pi i}{2}(h+k+l)}\right]
 \quad \text{for } h,k,l \text{ all even or all odd}$$

Two conditions, stacked. The fcc lattice requires $h,k,l$ unmixed; the basis then adds a
factor that vanishes when $h+k+l \equiv 2 \pmod 4$. So in the $[001]$ zone:

| $hkl$ | fcc | basis | result |
|---|---|---|---|
| 200 | all even ✓ | $h{+}k{+}l = 2 \equiv 2$ | **forbidden** |
| 220 | all even ✓ | $h{+}k{+}l = 4 \equiv 0$ | **allowed** |
| 400 | all even ✓ | $h{+}k{+}l = 4 \equiv 0$ | **allowed** |
| 420 | all even ✓ | $h{+}k{+}l = 6 \equiv 2$ | **forbidden** |

This is why the first allowed reflection of the square net is $\{220\}$ and not $\{200\}$,
and therefore why section 5's calibration used $\sqrt8/a$. Let us check it explicitly
rather than take it on trust.

In [ ]:
atoms = ase.build.bulk('Si', 'diamond', a=lattice_parameter_nm*10, cubic=True)
print(atoms)
print('fractional positions:\n', atoms.get_scaled_positions().round(3))
print(f'unit-cell volume: {atoms.cell.volume:.2f} A^3')

from diffsims.utils.lobato_scattering_params import ATOMIC_SCATTERING_PARAMS_LOBATO

def form_factor(symbol, g_length_invA):
    '''Electron atomic form factor f(g) in Angstrom, Lobato & Van Dyck (2014).'''
    a_b = np.array(ATOMIC_SCATTERING_PARAMS_LOBATO[symbol])
    a, b = a_b[:, 0], a_b[:, 1]
    g2 = np.asarray(g_length_invA, float)**2
    return np.sum(a * (2 + b*g2[..., None]) / (1 + b*g2[..., None])**2, axis=-1)

reciprocal_lattice = np.array(atoms.cell.reciprocal())     # 1/Angstrom

print('\n hkl    d (nm)    |F_hkl| (A)')
print('--------------------------------------')
for hkl in [(1,1,1), (2,0,0), (2,2,0), (3,1,1), (4,0,0), (4,2,0), (4,4,0)]:
    g_vec = np.dot(hkl, reciprocal_lattice)
    g_len = np.linalg.norm(g_vec)
    F = sum(form_factor(at.symbol, g_len) *
            np.exp(-2j*np.pi * (g_vec @ at.position)) for at in atoms)
    tag = '' if abs(F) > 1e-6 else '   <- forbidden'
    print(f' {"".join(map(str,hkl))}   {1/g_len/10:.4f}    {abs(F):8.2f}{tag}')

`200` and `420` come out at zero, as the table above predicts, and `220` survives. The
square net we measured in section 4 is the $\{220\}$ net, and the calibration stands.

Now the full pattern from `diffsims`, which packages this same kinematic physics.

In [ ]:
import diffpy.structure
from orix.crystal_map import Phase
from orix.quaternion import Rotation
from orix.vector import Vector3d
from diffsims.generators.simulation_generator import SimulationGenerator

# ---------------- INPUT ----------------
zone_axis         = [0, 0, 1]
reciprocal_radius = 14.0     # 1/nm, how far out to simulate
max_excitation    = 0.03     # 1/A
# ---------------------------------------

si_diffpy = diffpy.structure.Structure(
    atoms=[diffpy.structure.Atom('Si', p) for p in atoms.get_scaled_positions()],
    lattice=diffpy.structure.Lattice(*(atoms.cell.lengths()), *(atoms.cell.angles())))
phase = Phase(name='Si', space_group=227, structure=si_diffpy)
generator = SimulationGenerator(accelerating_voltage=acceleration_voltage_V/1000,
                                scattering_params='lobato')

rotation, _ = Rotation.from_align_vectors(
    Vector3d.zvector(), Vector3d(np.array(zone_axis, float)), return_rmsd=True)
simulation = generator.calculate_diffraction2d(
    phase, rotation=rotation,
    reciprocal_radius=reciprocal_radius/10,      # diffsims works in 1/Angstrom
    max_excitation_error=max_excitation)

coords = simulation.coordinates
g_sim = np.array(coords.data) * 10               # 1/A -> 1/nm
intensity = np.array(coords.intensity)
keep = (np.linalg.norm(g_sim, axis=1) > 1e-6) & (intensity > 1e-3*intensity.max())
g_sim, intensity = g_sim[keep], intensity[keep]

# recover conventional hkl from geometry, as in Part B
real_cell = np.array(atoms.cell)
g_crystal = (~rotation * Vector3d(g_sim/10)).data
hkl_sim = np.rint(g_crystal @ real_cell.T).astype(int)

print(f'{len(g_sim)} reflections simulated within {reciprocal_radius} 1/nm')
print('\nfirst few, by |g|:')
for i in np.argsort(np.linalg.norm(g_sim, axis=1))[:8]:
    print(f'  {str(hkl_sim[i]):16s} |g| = {np.linalg.norm(g_sim[i]):6.3f} 1/nm'
          f'   I = {intensity[i]/intensity.max():.3f}')

## 7. Overlay: indexing the experimental pattern

Everything is now in the same units. We place the simulated reflections on the detector
using the calibration from section 5 — scale, centre, rotation — and draw each one as a
disk of the *measured* convergence radius rather than a point.

The three numbers below are the only free parameters, and two of them were measured, not
guessed. Adjust `rotation_deg` if the disks sit on the right lattice but turned.

In [ ]:
# ---------------- INPUT ----------------
rotation_deg = rotation_measured_deg     # from section 5; nudge by hand to taste
# ---------------------------------------

theta = np.radians(rotation_deg)
R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
g_rotated = g_sim[:, :2] @ R.T

# reciprocal space (1/nm) -> detector pixels; image y axis points down
px = centre[0] + g_rotated[:, 0] / scale
py = centre[1] - g_rotated[:, 1] / scale

fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(np.log1p(img), cmap='gray')
inside = (px > 0) & (px < img.shape[1]) & (py > 0) & (py < img.shape[0])
relative_I = intensity / intensity.max()
for x, y, hkl, I in zip(px[inside], py[inside], hkl_sim[inside], relative_I[inside]):
    # line weight tracks the kinematic intensity, so strong and weak
    # reflections are not drawn as though they were equally expected
    ax.add_artist(plt.Circle((x, y), disk_radius_px, color='cyan', fill=False,
                             lw=0.6 + 2.2*I, alpha=0.35 + 0.6*I))
    label = ''.join(f'{v}' if v >= 0 else r'$\bar{%d}$' % -v for v in hkl)
    ax.annotate(label, (x, y), textcoords='offset points', xytext=(0, 0),
                ha='center', va='center', fontsize=9,
                color='yellow', alpha=0.45 + 0.55*I)
ax.add_artist(plt.Circle(centre, disk_radius_px, color='red', fill=False, lw=2))
ax.set_title(f'Si [001] CBED — kinematic simulation overlaid\n'
             f'scale {scale:.5f} 1/nm/px, rotation {rotation_deg:.1f} deg, '
             f'disk radius {convergence_mrad:.1f} mrad')
ax.set_xlim(0, img.shape[1]); ax.set_ylim(img.shape[0], 0); ax.axis('off')
plt.tight_layout()

print(f'{inside.sum()} simulated disks fall on the detector')

Compare this with the equivalent figure in Part B. There we overlaid *points* on diffuse
selected-area spots and judged the match by eye to within a few percent. Here the simulated
circles sit on sharp disk edges, and a misfit of a few pixels is visible — which is why CBED
is the method of choice when the geometry has to be right.

But notice how many of those circles are empty. The simulation offers every kinematically
allowed reflection out to $14\ \mathrm{nm}^{-1}$; the pattern does not show them. Rather
than argue about it from a figure, measure it. The test below compares the mean level inside
each predicted disk against the level *between* disks, which is the honest background.

In [ ]:
yy_i, xx_i = np.indices(img.shape)
flattened = ndimage.gaussian_filter(img / (ndimage.gaussian_filter(img, 320) + 1), 20)

def disk_level(p, r=90):
    if not (r < p[0] < img.shape[1]-r and r < p[1] < img.shape[0]-r):
        return None
    return flattened[(yy_i - p[1])**2 + (xx_i - p[0])**2 < r*r].mean()

shells, between = {}, []
for m in range(-3, 4):
    for n in range(-3, 4):
        level = disk_level(centre + m*a1 + n*a2)
        if level is None:
            continue
        shells.setdefault(round(np.hypot(*(m*a1 + n*a2))/g_first_px, 1), []).append(level)
        half = disk_level(centre + (m + 0.5)*a1 + n*a2)   # a position with no reflection
        if half is not None:
            between.append(half)

background_level = np.median(between)
print(f'reference level between disks = {background_level:.4f}\n')
print(f'{"shell":>7} {"n":>4} {"level":>9} {"excess":>10}')
for shell in sorted(shells):
    v = np.array(shells[shell])
    print(f'{shell:7.1f} {len(v):4d} {v.mean():9.4f} {100*(v.mean()/background_level-1):9.1f} %')

### What does the figure convey?

The numbers are blunt: **only the direct beam and the four $\{220\}$ disks carry real
intensity.** Shell 1.0 stands more than 100% above the between-disk background; every shell
beyond it sits within about 15% of it, which is to say at background. The $\{400\}$
reflections are allowed, they are simulated, their circles are drawn — and they are not
there.

That is not a failure of the calibration. It is three things the kinematic model does not
know about:

* **Convergence spreads intensity.** A reflection's counts are smeared over a disk of
  radius $\alpha/\lambda$ instead of concentrated in a spot. At $6.7\ \mathrm{mrad}$ these
  disks have about $10^4$ times the area of a selected-area spot, so anything weak sinks
  into the background.
* **Dynamical redistribution.** In a thick zone-axis crystal, intensity cycles between the
  direct beam and the strongly excited reflections. $\{220\}$ wins; the rest is starved.
  Kinematic intensities $|F_g|^2$ are simply the wrong model for which disks light up.
* **Exposure.** Two seconds, chosen so the direct beam does not saturate. The weak disks
  are below the noise floor, not absent in principle.

So the geometry is confirmed where there is signal to confirm it — five disks, each landing
within a few pixels of prediction — and the *intensities* are where kinematic theory starts
visibly failing. Which is exactly the right place to look next: the interesting part of this
pattern is not where the disks are, but what is happening **inside** them.

## 8. Inside the disk: where kinematic theory ends

Position within a disk maps to excitation error, so a line profile across a disk, along
$\vec g$, **is** a rocking curve. The mapping needs care, and it is worth doing properly
because the obvious guess is wrong.

For an incident beam with transverse wavevector $\vec k_t$ measured from the zone axis,

$$s_g = -\frac{2\vec K_0\!\cdot\!\vec g + g^2}{2K_0}
      \;\simeq\; -\lambda g\left(\frac{g}{2} + \vec k_t\!\cdot\!\hat g\right)$$

Inside the disk for reflection $\vec g$, displacement from the *disk centre* along $\hat g$
is exactly $\vec k_t\!\cdot\!\hat g$. So:

* at the disk centre ($\vec k_t = 0$, the on-axis ray) $s_g = -\lambda g^2/2$, which is
  **not zero**;
* $s_g = 0$ occurs at $\vec k_t\!\cdot\!\hat g = -g/2$ — displaced from the disk centre
  *towards the direct beam* by $g/2$.

For our pattern $g/2 = 2.60\ \mathrm{nm^{-1}}$ and the disk radius is
$2.67\ \mathrm{nm^{-1}}$, so the exact-Bragg line falls almost precisely on the **inner edge**
of the disk. That is not a coincidence: it is the same statement as $k_\alpha \simeq g/2$,
the Kossel–Möllenstedt boundary from section 3. Putting $s_g = 0$ at the disk centre — the
natural-looking shortcut — would misplace the whole rocking curve by a full disk radius.

Kinematic theory predicts

$$I_g^{\text{kin}}(s) \;=\; \left(\frac{\pi t}{\xi_g}\right)^{2}
  \mathrm{sinc}^2\!\left(t\,s\right),
  \qquad \mathrm{sinc}(x) = \frac{\sin \pi x}{\pi x}$$

which diverges as $t^2$ at $s=0$ — an unphysical prediction that the crystal scatters more
than 100% of the beam into one reflection for a thick enough specimen. Two-beam dynamical
theory fixes this by letting intensity flow *back* from $\vec g$ into the direct beam. The
result is the same expression with $s$ replaced by an effective excitation error:

$$I_g^{\text{dyn}}(s) = \left(\frac{\pi t}{\xi_g}\right)^{2}
  \mathrm{sinc}^2\!\left(t\, s_{\text{eff}}\right),
  \qquad s_{\text{eff}} = \sqrt{s^2 + \xi_g^{-2}}$$

The intensity now oscillates between 0 and 1 as thickness grows — **Pendellösung** — and
the fringe minima are *unevenly* spaced in $s$. That uneven spacing is the signal: it
depends on $t$ and $\xi_g$ separately, which is what makes thickness measurable from a
single CBED disk.

The extinction distance that sets the scale is

$$\xi_g = \frac{\pi V_c}{\lambda\,\gamma\,F_g},
   \qquad \gamma = 1 + \frac{eV}{m_e c^2}$$

The factor $\gamma$ is easy to drop and worth a sentence. Tabulated electron scattering
factors — Lobato's included — are *non-relativistic*. At 200 kV the electron is heavy
enough that $\gamma = 1.39$, so leaving it out makes every extinction distance 39% too
long. Part B never needed it, because kinematic spot *positions* do not depend on it; the
moment we ask about intensities inside a disk, it matters.

In [ ]:
hkl_profile = (2, 2, 0)
g_vec = np.dot(hkl_profile, reciprocal_lattice)
g_len_invA = np.linalg.norm(g_vec)
F_g = abs(sum(form_factor(at.symbol, g_len_invA) *
              np.exp(-2j*np.pi * (g_vec @ at.position)) for at in atoms))

V_c_A3 = atoms.cell.volume
lambda_A = wave_length_m * 1e10
gamma = 1 + const.e*acceleration_voltage_V / (const.m_e * const.c**2)
xi_g_A = np.pi * V_c_A3 / (lambda_A * gamma * F_g)

print(f'{hkl_profile}:  |F_g| = {F_g:.2f} A,  V_c = {V_c_A3:.2f} A^3,  gamma = {gamma:.4f}')
print(f'extinction distance xi_g = {xi_g_A/10:.1f} nm')
print(f'  without the relativistic factor it would be {gamma*xi_g_A/10:.1f} nm -- 39% too long')
print('  (the tabulated 75.7 nm for Si {220} is the 100 kV value; scaled to 200 kV it is 96 nm)')

g_220_invnm = g_len_invA * 10
lambda_nm = wave_length_m * 1e9
ds_dk = lambda_nm * g_220_invnm              # ds_g per unit displacement in 1/nm
s_at_disk_centre = -lambda_nm * g_220_invnm**2 / 2

print(f'\nacross a {{220}} disk:  ds_g/dk = {ds_dk:.5f}')
print(f'  s_g at the disk centre          = {s_at_disk_centre:+.4f} 1/nm  (NOT zero)')
print(f'  s_g = 0 at displacement         = {-g_220_invnm/2:+.3f} 1/nm from the centre')
print(f'  disk radius                     = {convergence_invnm:+.3f} 1/nm')
print(f'  -> exact Bragg sits at {100*(g_220_invnm/2)/convergence_invnm:.0f}% of the disk '
      f'radius, i.e. on the inner edge')
print(f'  s_g spans {-ds_dk*(g_220_invnm/2 - convergence_invnm):+.4f} to '
      f'{-ds_dk*(g_220_invnm/2 + convergence_invnm):+.4f} 1/nm across the disk')

Now extract the real profile. We take the brightest first-order disk and read a line
across it, through its centre, along $\vec g$.

In [ ]:
# pick the first-order disk with the most signal
first_order = [centre + a1, centre - a1, centre + a2, centre - a2]
brightness = [ndimage.gaussian_filter(img, 10)[int(p[1]), int(p[0])] for p in first_order]
disk_centre = first_order[int(np.argmax(brightness))]
g_direction = (disk_centre - centre) / np.linalg.norm(disk_centre - centre)

t_px = np.linspace(-disk_radius_px, disk_radius_px, 400)   # stay inside the disk
line_x = disk_centre[0] + g_direction[0]*t_px
line_y = disk_centre[1] + g_direction[1]*t_px
# average over a short perpendicular window to beat down noise
perp = np.array([-g_direction[1], g_direction[0]])
stack = [ndimage.map_coordinates(img, [line_y + perp[1]*o, line_x + perp[0]*o], order=1)
         for o in np.arange(-25, 26, 2.0)]
line_profile = np.mean(stack, axis=0)

# t_px > 0 points AWAY from the direct beam, so k_t.ghat = +t_px*scale
s_g = -ds_dk * (g_220_invnm/2 + t_px*scale)

fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
ax[0].imshow(np.log1p(img), cmap='gray')
ax[0].plot(line_x, line_y, 'r-', lw=2)
ax[0].add_artist(plt.Circle(disk_centre, disk_radius_px, color='cyan', fill=False, lw=1.3))
ax[0].plot(*centre, 'y+', ms=14, mew=2)
ax[0].set_xlim(disk_centre[0]-320, disk_centre[0]+320)
ax[0].set_ylim(disk_centre[1]+320, disk_centre[1]-320)
ax[0].set_title('a {220} disk and the profile line\n(yellow + = direct beam)')
ax[1].plot(s_g, line_profile, 'k-', lw=1.2)
ax[1].axvline(0, color='b', ls=':', lw=1.5)
ax[1].text(0, line_profile.max()*0.97, ' exact Bragg,\n $s_g=0$', color='b',
           fontsize=9, va='top')
ax[1].set_xlabel('excitation error $s_g$ (1/nm)')
ax[1].set_ylabel('counts')
ax[1].set_title('the rocking curve, measured in one exposure')
ax[1].grid(alpha=.3)
plt.tight_layout()

print(f's_g runs from {s_g[0]:+.4f} to {s_g[-1]:+.4f} 1/nm across the disk')
print(f'brightest point of the profile is at s_g = '
      f'{s_g[np.argmax(ndimage.gaussian_filter1d(line_profile, 5))]:+.4f} 1/nm')

There is the rocking curve — oscillations that a uniform kinematic disk simply does not
have, acquired in a single two-second exposure without tilting anything.

Note where the fringes are. Exact Bragg sits at the inner edge, and the oscillations run
*away* from it into increasingly negative $s_g$, with the spacing tightening as $|s_g|$
grows. That tightening is the dynamical signature: kinematic minima are evenly spaced in
$s$, dynamical ones are not.

Below, both models are drawn against the measurement. The thickness is a free parameter:
change `thickness_nm` and watch the fringe count. This is a *demonstration* of the physics,
not a production thickness measurement — the honest method is a Kelly plot, which uses the
positions of several minima to solve for $t$ and $\xi_g$ simultaneously.

In [ ]:
# ---------------- INPUT ----------------
thickness_nm = 250.0
# ---------------------------------------

xi_nm = xi_g_A / 10
s_model = np.linspace(s_g.min(), s_g.max(), 600)

def sinc2(x):
    return np.sinc(x)**2          # numpy sinc is sin(pi x)/(pi x)

I_kin = (np.pi*thickness_nm/xi_nm)**2 * sinc2(thickness_nm*s_model)
s_eff = np.sqrt(s_model**2 + 1/xi_nm**2)
I_dyn = (np.pi*thickness_nm/xi_nm)**2 * sinc2(thickness_nm*s_eff)

fig, ax = plt.subplots(figsize=(10, 5.5))
norm = lambda y: (y - y.min())/(y.max() - y.min())
ax.plot(s_g, norm(line_profile), 'k-', lw=1.6, label='measured')
ax.plot(s_model, norm(I_kin), 'r--', lw=1.2, label=f'kinematic, t = {thickness_nm:.0f} nm')
ax.plot(s_model, norm(I_dyn), 'b-', lw=1.4,
        label=f'two-beam dynamical, t = {thickness_nm:.0f} nm, $\\xi_g$ = {xi_nm:.0f} nm')
ax.set_xlabel('excitation error $s_g$ (1/nm)'); ax.set_ylabel('normalised intensity')
ax.set_title('inside a {220} disk: kinematic vs two-beam dynamical')
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout()

print(f'kinematic minima are evenly spaced in s:   ds = 1/t = {1/thickness_nm:.4f} 1/nm')
print('dynamical minima are NOT evenly spaced — they satisfy  t*sqrt(s^2 + 1/xi^2) = n,')
print('which is what lets a Kelly plot separate t from xi_g.')

### What is the accuracy?

**The geometry is good to about a percent.** The lattice fit in section 4 had an rms
residual of a few pixels on a 300-pixel spacing, and every disk with real signal lands
within a few pixels of prediction. The limiting systematic is the elliptical distortion we
measured and then averaged over — a proper job would fit the ellipse and correct it.

**The dynamical part does not fit, and it should not.** Look at the comparison honestly: the
two-beam curve has roughly the right fringe spacing, but the measured maximum is not at
$s_g = 0$, the measured curve is lopsided, and the fringe contrast is wrong. Two-beam theory
assumes exactly one reflection is excited. We are at a $[001]$ zone axis, where four
$\{220\}$ reflections are strongly excited at once and feeding each other — which is, after
all, why the pattern looks the way it does. The correct treatment is a many-beam Bloch-wave
or multislice calculation (`abTEM`, `py4DSTEM`).

So treat the thickness above as an order of magnitude. If you need a number that carries
weight, tilt to a two-beam condition — where this formalism is actually valid — or simulate
many-beam. The value of the comparison is that it shows you precisely *which* assumption
broke, which is more useful than a curve that happened to fit.

### Why this matters beyond one pattern

The disks here are a *map* of excitation error, which means a CBED pattern is the single-probe
limit of a **4D-STEM** dataset: scan this probe and you record one of these at every pixel.
The centre-of-mass shift of the direct disk is then the differential phase contrast signal,
and the fringes carry thickness at every scan point. The analysis in this notebook is the
per-probe kernel of that much larger experiment — which is where the CITEAM 4D-STEM module
picks up.

## 9. Data-management epilogue: a dataset that corrects its own file

Part B took an orphan TIF with *no* metadata and gave it some. This file is the harder
case: it arrived with plenty of metadata, and some of it was wrong. Deleting the wrong
values would be its own kind of dishonesty — the next person would have no way of knowing
the file had ever disagreed with the specimen.

So we write a dataset that carries the corrected calibration **and** a record of what it
corrected, and why.

In [ ]:
for ax_, name in zip(s.axes_manager.signal_axes, ('kx', 'ky')):
    ax_.name  = name
    ax_.units = '1/nm'
    ax_.scale = scale
s.axes_manager.signal_axes[0].offset = -centre[0] * scale
s.axes_manager.signal_axes[1].offset = -centre[1] * scale

s.metadata.set_item('General.title', 'CBED pattern, Si [001]')
s.metadata.set_item('Acquisition_instrument.TEM.beam_energy',
                    acceleration_voltage_V/1000)
s.metadata.set_item('Acquisition_instrument.TEM.camera_length',
                    float(camera_length_true_m*1e3))
s.metadata.set_item('Acquisition_instrument.TEM.convergence_angle',
                    float(convergence_mrad))
s.metadata.set_item('Sample.description',
                    'Silicon, [001] zone axis (NOT the polymer named in the DM3 '
                    'Specimen field, which is stale from an earlier session)')
s.metadata.set_item('General.notes', '; '.join([
    f'Calibration refined against Si {{220}} at a={lattice_parameter_nm*10:.4f} A: '
    f'{scale:.6f} 1/nm per px',
    f'DM3 axis calibration was {0.4032:.4f} nm/px (REAL space) and has been replaced',
    f'DM3 Operation Mode read IMAGING; this is a diffraction pattern',
    f'Nominal camera length {camera_length_nominal_m*1e3:.0f} mm disagrees with the '
    f'specimen by {100*(dg_nominal/scale-1):+.0f}%; refined value {camera_length_true_m*1e3:.0f} mm',
    f'Convergence {convergence_mrad:.1f} mrad measured from the disk radius '
    f'(file name says 3.3 mrad, which is the filter illumination angle)',
    f'DM3 Specimen field names a block copolymer and is stale',
    f'Processed with CITEAM 2027 CBED notebook v{__notebook_version__}']))

s.save('Si_CBED_001_calibrated.hspy', overwrite=True)

s2 = hs.load('Si_CBED_001_calibrated.hspy')
print(s2.axes_manager)
print(s2.metadata)

The `.hspy` now travels with a reciprocal-space calibration tied to a known crystal, the
instrument parameters that survived scrutiny, and — in `General.notes` — a plain statement
of every value it had to overrule. A reader who disagrees with our indexing can find the
original numbers in `original_metadata`, which `hspy` preserves untouched, and redo the
work.

That is the difference between a file and a dataset. The file was not short of
information; it was short of *trustworthy* information, and nothing in it said which was
which.

The next step is the same as in Part B: a DOI-minted deposit (Zenodo via a GitHub release,
or the Materials Data Facility) of the notebook together with the calibrated data.

### Exercise (Part E of the module)

1. Re-run section 4 with `smooth_disk` doubled and halved. How far does the refined scale
   move? That spread is your real calibration uncertainty — quote it.
2. The net is not exactly square. Fit an ellipse to the first-order disks instead of
   averaging $|a_1|$ and $|a_2|$, correct the distortion, and see whether the overlay in
   section 7 improves at the pattern edges.
3. Section 7 found no $\{400\}$ disk above background. Compute $\xi_{400}$ and, with the
   thickness from section 8, work out how much weaker the $\{400\}$ rocking curve should be
   than $\{220\}$. Is the two-beam estimate enough to explain a disk you cannot see, or do
   you need the dynamical argument as well?
4. Using the two-beam expression, work out the thickness at which the $\{220\}$ disk centre
   first goes *dark*. What does that tell you about using a single disk to measure thickness?

---

### Attribution and references

* Pedagogical structure adapted from G. Duscher & S. Kalinin, *Analyzing CBED Pattern*,
  MLSTEM 2026, University of Tennessee, Knoxville —
  [github.com/gduscher/MLSTEM2026](https://github.com/gduscher/MLSTEM2026).
  Data file `CBED_375mm_alpha-3.3_beta9.6_Condensor10um_Filter.dm3` from that repository's
  `example_data`, recorded on a Zeiss Libra 200 MC, 2017-09-27.
* Theory notation follows the CITEAM lecture *Electron Diffraction Patterns from Al* and
  Part B of this module.
* Form factors: I. Lobato & D. Van Dyck, *Acta Cryst.* **A70** (2014) 636, as shipped with
  `diffsims`.
* Two-beam dynamical theory and extinction distances: P. Hirsch et al., *Electron
  Microscopy of Thin Crystals*, Butterworths (1965), ch. 10; D. B. Williams & C. B. Carter,
  *Transmission Electron Microscopy*, Springer (2009), ch. 23–27.
* CBED thickness determination (Kelly plot): P. M. Kelly et al., *Phys. Status Solidi A*
  **31** (1975) 771.
* Software: HyperSpy, diffsims, orix, ASE — please cite their respective papers when
  publishing results.
